# 02 — Análise exploratória

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Tabela:** `ecommerce_itens_pedido`

Todo cálculo roda no Spark; os gráficos usam só o resultado já agregado. A seção 5 resume o que a
tabela diz para o negócio.

| # | Regra verificada (seção 3) |
|---|---|
| R1 | Nenhum nulo por coluna |
| R2 | `id_item_pedido` sem nulo e sem duplicata |
| R3 | `quantidade > 0` |
| R4 | `preco_unitario > 0` e no máximo 60% abaixo do `preco_lista` do catálogo |
| R5 | `desconto_aplicado` entre 0 e `preco_unitario` |
| R6 | todo `id_pedido` existe em `ecommerce_pedidos` |

**Indicadores (seção 4):** SKUs mais vendidos, distribuição de quantidade, itens por pedido, receita bruta x
líquida e peso do desconto.

In [ ]:
%run ./01_extracao_itens_pedido

## 1. O que é esta tabela

Cada linha é **um produto dentro de um pedido**: o pedido `id_pedido` levou `quantidade` unidades do `sku`,
a `preco_unitario` cada, com `desconto_aplicado` por unidade. É o detalhe de `ecommerce_pedidos` e a
ponte entre pedidos e o catálogo (`ecommerce_produtos`).

| Coluna | Significado |
|---|---|
| `id_item_pedido` | identificador do item (chave primária) |
| `id_pedido` | pedido ao qual o item pertence → `ecommerce_pedidos` |
| `sku` | produto vendido → `ecommerce_produtos` |
| `quantidade` | unidades do produto no pedido |
| `preco_unitario` | preço cobrado por unidade |
| `desconto_aplicado` | desconto por unidade |
| `dt_lote` | *(linhagem, do `01`)* lote do gerador em que o item chegou |

Receita bruta do item = `preco_unitario × quantidade`; líquida = `(preco_unitario − desconto_aplicado) × quantidade`.

In [ ]:
import matplotlib.pyplot as plt

AZUL, AZUL2, EIXO, FUNDO = "#1F4E8C", "#5B8FC9", "#595959", "#F7F7F7"
plt.rcParams.update({"font.family": "Arial", "axes.edgecolor": EIXO, "axes.labelcolor": EIXO, "xtick.color": EIXO,
                     "ytick.color": EIXO, "text.color": "#262626", "figure.facecolor": FUNDO, "axes.facecolor": FUNDO})


def barras(pdf, x, y, titulo, rotulo, horizontal=False):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    if horizontal:
        ax.barh(pdf[x].astype(str), pdf[y], color=AZUL); ax.invert_yaxis(); ax.set_xlabel(rotulo)
    else:
        ax.bar(pdf[x].astype(str), pdf[y], color=AZUL); ax.set_ylabel(rotulo); plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_title(titulo, loc="left", fontweight="bold"); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()


itens = df_itens.withColumn("receita_bruta", F.col("preco_unitario") * F.col("quantidade")) \
                .withColumn("receita_liquida", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
total = itens.count()
print(f"Linhas: {total} | colunas: {len(df_itens.columns)}")

## 2. Chegada dos lotes

Itens e receita bruta por dia de lote: mostra quando as janelas de tempo real estiveram abertas e com que volume.

In [ ]:
por_dia = itens.groupBy(F.to_date("dt_lote").alias("dia")).agg(F.countDistinct("dt_lote").alias("lotes"), F.count("*").alias("itens"),
                                                              F.countDistinct("id_pedido").alias("pedidos"), F.round(F.sum("receita_bruta"), 2).alias("receita_bruta")).orderBy("dia")
display(por_dia)
barras(por_dia.toPandas(), "dia", "itens", "Itens recebidos por dia de lote", "itens")

## 3. Qualidade dos dados

### 3.1 Nulos, chave e cardinalidade

Não aplicamos `dropDuplicates` na extração: itens são fato incremental, e duplicata aqui seria um achado
sobre a origem (reenvio de arquivo), não algo a apagar em silêncio. A Silver deduplica (regra T5).

In [ ]:
nulos = itens.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_itens.columns]).first().asDict()
distintos = itens.select([F.countDistinct(c).alias(c) for c in df_itens.columns]).first().asDict()
display(spark.createDataFrame([(c, int(nulos[c]), int(distintos[c])) for c in df_itens.columns], ["coluna", "nulos", "distintos"]))

pk_nulos, pk_dup = nulos[CHAVE_PRIMARIA], total - distintos[CHAVE_PRIMARIA]
print(f"{CHAVE_PRIMARIA}: nulos {pk_nulos} | duplicados {pk_dup} -> {'SAUDAVEL' if pk_nulos == 0 and pk_dup == 0 else 'ATENCAO'}")
if pk_dup:
    display(itens.groupBy(CHAVE_PRIMARIA).count().filter("count > 1").orderBy(F.desc("count")))

### 3.2 Regras de negócio

Para R4 e R6 a análise lê também `ecommerce_produtos` e `ecommerce_pedidos` do mesmo lake. É só
exploração: na Silver essas checagens usam a **Silver** de pedidos e de produtos, nunca o `raw`.
O catálogo repete os produtos a cada lote, por isso fica uma linha por `sku`, a do lote mais recente.

| Regra | Por que importa |
|---|---|
| `quantidade <= 0` | zera ou inverte a receita do item |
| `preco_unitario <= 0` | item de graça ou com preço negativo distorce receita |
| preço > 60% abaixo do `preco_lista` | indica erro na busca do preço no catálogo |
| `desconto_aplicado > preco_unitario` | desconto impossível: o item sairia com valor negativo |
| `desconto_aplicado < 0` | desconto negativo vira acréscimo escondido |
| `id_pedido` sem pedido | item órfão: não se liga a cliente, pagamento nem status |

In [ ]:
from pyspark.sql import Window

caminho_de = lambda t: CAMINHO_ORIGEM.replace(f"/{TABELA}.parquet", f"/{t}.parquet")
produtos = com_lote(ler_origem(caminho_de("ecommerce_produtos"), "sku string, preco_lista double"))
catalogo = produtos.withColumn("_rn", F.row_number().over(Window.partitionBy("sku").orderBy(F.desc("dt_lote")))).filter("_rn = 1").select("sku", "preco_lista")
pedidos = ler_origem(caminho_de("ecommerce_pedidos"), "id_pedido bigint").select("id_pedido").distinct().withColumn("tem_pedido", F.lit(True))

base = itens.join(catalogo, "sku", "left").join(pedidos, "id_pedido", "left")
REGRAS = {
    "quantidade <= 0": F.col("quantidade") <= 0,
    "preco_unitario <= 0": F.col("preco_unitario") <= 0,
    "preco > 60% abaixo do catalogo": (F.col("preco_lista") > 0) & (F.col("preco_unitario") < 0.4 * F.col("preco_lista")),
    "desconto > preco_unitario": F.col("desconto_aplicado") > F.col("preco_unitario"),
    "desconto negativo": F.col("desconto_aplicado") < 0,
    "id_pedido sem pedido": F.col("tem_pedido").isNull(),
}
base = base.withColumn("regras_violadas", F.concat_ws(", ", *[F.when(F.coalesce(c, F.lit(False)), F.lit(n)) for n, c in REGRAS.items()]))
violacoes = base.agg(*[F.sum(F.coalesce(c, F.lit(False)).cast("int")).alias(n) for n, c in REGRAS.items()]).first().asDict()
display(spark.createDataFrame([(n, int(q), round(q / total * 100, 2)) for n, q in violacoes.items()], ["regra_violada", "itens", "pct"]))
incoerentes = base.filter(F.col("regras_violadas") != "")
print(f"Itens com ao menos uma violacao: {incoerentes.count()} de {total} | SKUs fora do catalogo: {base.filter(F.col('preco_lista').isNull()).count()}")
display(incoerentes.select(CHAVE_PRIMARIA, "id_pedido", "sku", "quantidade", "preco_unitario", "preco_lista", "desconto_aplicado", "regras_violadas").limit(50))

## 4. O que os itens dizem sobre o negócio

Os indicadores abaixo usam só os itens que **não** violam nenhuma regra, para que preço zerado ou
desconto impossível não distorçam receita e ranking.

### 4.1 SKUs mais vendidos e quantidade por item

In [ ]:
ok = base.filter(F.col("regras_violadas") == "")
top = ok.groupBy("sku").agg(F.sum("quantidade").alias("unidades"), F.round(F.sum("receita_liquida"), 2).alias("receita_liquida")).orderBy(F.desc("unidades")).limit(10)
display(top)
barras(top.toPandas(), "sku", "unidades", "Top 10 SKUs por unidades vendidas", "unidades", horizontal=True)
qtd = itens.groupBy("quantidade").count().orderBy("quantidade")
barras(qtd.toPandas(), "quantidade", "count", "Itens por quantidade de unidades (todos os itens)", "itens")

### 4.2 Itens por pedido

In [ ]:
por_pedido = ok.groupBy("id_pedido").agg(F.count("*").alias("itens"), F.sum("quantidade").alias("unidades"))
display(por_pedido.agg(F.round(F.avg("itens"), 2).alias("itens_medio"), F.percentile_approx("itens", 0.5).alias("itens_mediana"),
                       F.max("itens").alias("itens_max"), F.round(F.avg("unidades"), 2).alias("unidades_medio"),
                       F.round(F.avg((F.col("itens") < 3).cast("int")) * 100, 1).alias("pct_pedidos_com_menos_de_3_itens")))
barras(por_pedido.groupBy("itens").count().orderBy("itens").toPandas(), "itens", "count", "Pedidos por número de itens", "pedidos")

### 4.3 Receita bruta x líquida e peso do desconto

A planilha define alerta quando o desconto passa de 25% da receita bruta.

In [ ]:
rec = ok.agg(F.round(F.sum("receita_bruta"), 2).alias("receita_bruta"), F.round(F.sum("receita_liquida"), 2).alias("receita_liquida"),
             F.round(F.avg((F.col("desconto_aplicado") > 0).cast("int")) * 100, 1).alias("pct_itens_com_desconto")).first()
pct = (1 - rec["receita_liquida"] / rec["receita_bruta"]) * 100
print(f"Receita bruta: R$ {rec['receita_bruta']:,.2f} | liquida: R$ {rec['receita_liquida']:,.2f} | desconto: {pct:.1f}% da bruta | itens com desconto: {rec['pct_itens_com_desconto']}%")
pdia = ok.groupBy(F.to_date("dt_lote").alias("dia")).agg(F.sum("receita_bruta").alias("bruta"), F.sum("receita_liquida").alias("liquida")).orderBy("dia").toPandas()
fig, ax = plt.subplots(figsize=(9, 3.5)); x = range(len(pdia))
ax.bar([i - 0.2 for i in x], pdia["bruta"], 0.4, color=AZUL, label="bruta"); ax.bar([i + 0.2 for i in x], pdia["liquida"], 0.4, color=AZUL2, label="líquida")
ax.set_xticks(list(x), pdia["dia"].astype(str)); ax.set_ylabel("R$"); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Receita bruta x líquida por dia de lote", loc="left", fontweight="bold"); plt.tight_layout(); plt.show()

## 5. Resumo para o negócio

> **Base desta leitura:** 353 lotes de tempo real entre 21/09 e 08/10/2026, **13.325 itens** de **2.222
> pedidos** e **3.038 SKUs**. Se novas janelas forem abertas, as células acima se atualizam; revisar os
> números deste texto depois de reexecutar.

**O que a tabela é.** `ecommerce_itens_pedido` detalha o que foi vendido em cada pedido: produto,
quantidade, preço e desconto. É a única tabela que diz **quais produtos** saem e a que preço, e por isso
sustenta ranking de produtos, impacto de desconto e tamanho da cesta.

**Integridade.** Nenhum nulo, chave `id_item_pedido` única e todos os itens com pedido correspondente,
chegando no mesmo lote do pedido. Só 2 itens têm SKU fora do catálogo.

**Incoerências (cerca de 2.400 itens, 18%):**
- **469 itens com quantidade zero** (3,5%): somem da receita sem deixar rastro no pedido.
- **191 itens com preço zero ou negativo** (82 com zero, mínimo −R$ 299,99).
- **Cerca de 1.500 itens com preço mais de 60% abaixo do catálogo** (11%), o maior grupo; o número exato
  depende da versão do catálogo usada. A regra do negócio
  trata isso como erro de busca de preço; vale confirmar se não há promoção legítima nesse grupo.
- **634 itens com desconto maior que o preço unitário** (4,8%), desconto impossível; e 11 com desconto negativo.

**Leitura de negócio (itens válidos):**
- **Cesta:** em média 5 itens e 8 unidades por pedido (mediana de 4 itens); a maior parte dos itens sai com
  1 unidade. Cerca de 21% dos pedidos têm menos de 3 itens, o limite de alerta do KPI N10.
- **Desconto:** 15% dos itens válidos têm desconto, que pesa 1,8% da receita bruta (R$ 581 mil bruta,
  R$ 571 mil líquida), longe do limite de alerta de 25%. Nos dados brutos o peso chega a 5,9%, puxado
  pelos descontos impossíveis.
- **Produtos:** refrigerantes, água mineral, leite UHT e sabão líquido lideram em unidades: perfil de
  supermercado, compra de reposição. Nenhum SKU concentra a venda (o líder tem 50 unidades em 13 mil itens).

**O que muda na Silver:** quarentena para quantidade, preço, catálogo e desconto (T3, T4, N9), espera para
item sem pedido (T2) e deduplicação por `id_item_pedido` (T5).